In [1]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy
import warnings
import pickle
import pandas as pd
import lib_analise 
warnings.filterwarnings('ignore')

In [2]:
info_modelo = lib_analise.get_info_modelo()  # para garantir que a função está carregada da lib_analise correta
datasets = lib_analise.get_dataset_analise(analise_ganho_de_informacao=False)
lib_analise.print_informacao_analise()

Nome do dataset:  svm
X_train.shape (32259, 122)
X_test.shape (24194, 122)
X_val.shape (24195, 122)
X_train_scaled.shape None
X_test_scaled.shape None
X_val_scaled.shape None
classes_mapping {'interf': np.int64(0), 'normal': np.int64(1)}
features_ganho_informacao ['mean_os_cpu_ctx_switches', 'mean_os_cpu_guest', 'mean_os_cpu_guest_nice', 'mean_os_cpu_idle', 'mean_os_cpu_interrupts', 'mean_os_cpu_iowait', 'mean_os_cpu_irq', 'mean_os_cpu_nice', 'mean_os_cpu_soft_interrupts', 'mean_os_cpu_softirq', 'mean_os_cpu_steal', 'mean_os_cpu_syscalls', 'mean_os_cpu_system', 'mean_os_cpu_user', 'mean_os_disk_discard_io', 'mean_os_disk_discard_merges', 'mean_os_disk_discard_sectors', 'mean_os_disk_discard_ticks', 'mean_os_disk_in_flight', 'mean_os_disk_io_ticks', 'mean_os_disk_read_io', 'mean_os_disk_read_merge', 'mean_os_disk_read_sectors', 'mean_os_disk_read_ticks', 'mean_os_disk_time_in_queue', 'mean_os_disk_write_io', 'mean_os_disk_write_merge', 'mean_os_disk_write_sectors', 'mean_os_disk_write_t

In [3]:
# Cálculo do Mutual Information usando scikit-learn
print("Calculando Mutual Information...")

# Prepara os dados para o mutual information
# Remove valores NaN
def calcular_mutual_information(X,y):
    mask = ~(X.isna().any(axis=1) | pd.isna(y))
    X_clean = X[mask]
    y_clean = y[mask]

    # Para features categóricas, usa mutual_info_classif diretamente
    # Para features contínuas, também funciona bem
    try:
        # Calcula mutual information
        mi_scores = mutual_info_classif(X_clean, y_clean, random_state=42)
        
        # Cria DataFrame com os resultados
        mi_results = pd.DataFrame({
            'Feature': X_clean.columns,
            'Mutual_Information': mi_scores
        })
        
        # Ordena por mutual information
        mi_results = mi_results.sort_values('Mutual_Information', ascending=False)
        
        print("\nMutual Information por Feature (Top 15):")
        print(mi_results.head(15))
        
    except Exception as e:
        print(f"Erro no cálculo do Mutual Information: {e}")
        print("Tentando com encoding das variáveis categóricas...")
        
        # Se houver erro, tenta fazer encoding das variáveis categóricas
        X_encoded = X_clean.copy()
        label_encoders = {}
        
        for column in X_encoded.columns:
            if X_encoded[column].dtype == 'object':
                le = LabelEncoder()
                X_encoded[column] = le.fit_transform(X_encoded[column].astype(str))
                label_encoders[column] = le
        
        # Tenta novamente
        mi_scores = mutual_info_classif(X_encoded, y_clean, random_state=42)
        
        mi_results = pd.DataFrame({
            'Feature': X_encoded.columns,
            'Mutual_Information': mi_scores
        })
        
        mi_results = mi_results.sort_values('Mutual_Information', ascending=False)
    return mi_results

mi_results = calcular_mutual_information(datasets['X_train'], datasets['y_train'])

print("\nMutual Information por Feature (Top 15):")
print(mi_results)




Calculando Mutual Information...

Mutual Information por Feature (Top 15):
                          Feature  Mutual_Information
41         mean_os_net_bytes_sent            0.657781
46    mean_os_net_num_connections            0.531574
52        mean_process_cpu_system            0.404249
57  mean_process_disk_write_bytes            0.391475
53          mean_process_cpu_user            0.364710
56   mean_process_disk_read_count            0.354320
59  mean_process_disk_write_count            0.338401
55   mean_process_disk_read_chars            0.314235
48       mean_os_net_packets_sent            0.285750
40         mean_os_net_bytes_recv            0.272257
58  mean_process_disk_write_chars            0.246754
67      mean_process_net_rx_bytes            0.239648
47       mean_os_net_packets_recv            0.227896
96       mean_container_mem_cache            0.225081
12             mean_os_cpu_system            0.206797

Mutual Information por Feature (Top 15):
                   

In [4]:
# Análise da distribuição por faixas de valores
def analyze_distribution_ranges(values, name):
    """Analisa a distribuição de valores em diferentes faixas"""
    print(f"\n🔍 ANÁLISE DE DISTRIBUIÇÃO - {name}:")
    
    # Definir faixas de análise
    ranges = [
        (0.0, 0.001, "Muito Baixa"),
        (0.001, 0.01, "Baixa"),
        (0.01, 0.05, "Moderada"),
        (0.05, 0.1, "Alta"),
        (0.1, float('inf'), "Muito Alta")
    ]
    
    total_values = len(values.dropna())
    for min_val, max_val, label in ranges:
        if max_val == float('inf'):
            count = (values >= min_val).sum()
            range_str = f">= {min_val}"
        else:
            count = ((values >= min_val) & (values < max_val)).sum()
            range_str = f"[{min_val}, {max_val})"
        
        percentage = (count / total_values) * 100
        print(f"   • {label:12s} {range_str:12s}: {count:3d} features ({percentage:5.1f}%)")

analyze_distribution_ranges(mi_results['Mutual_Information'], "MUTUAL INFORMATION")



🔍 ANÁLISE DE DISTRIBUIÇÃO - MUTUAL INFORMATION:
   • Muito Baixa  [0.0, 0.001):  39 features ( 32.0%)
   • Baixa        [0.001, 0.01):  24 features ( 19.7%)
   • Moderada     [0.01, 0.05):  16 features ( 13.1%)
   • Alta         [0.05, 0.1) :   8 features (  6.6%)
   • Muito Alta   >= 0.1      :  35 features ( 28.7%)


In [5]:
# Adicionar estatísticas resumidas no MI
def calculate_mi_statistics(mi_results):
    """Calcula estatísticas resumidas do Mutual Information"""  
    mi_stats = pd.DataFrame({
    'Estatística': ['Total Features', 'Média', 'Mediana', 'Desvio Padrão',
                    'Mínimo', 'Máximo', 'Q1 (25%)', 'Q3 (75%)', 'Q90 (Top 10%)'],
    'Valor': [
        len(mi_results),
        mi_results['Mutual_Information'].mean(),
        mi_results['Mutual_Information'].median(),
        mi_results['Mutual_Information'].std(),
        mi_results['Mutual_Information'].min(),
        mi_results['Mutual_Information'].max(),
        mi_results['Mutual_Information'].quantile(0.25),
        mi_results['Mutual_Information'].quantile(0.75),
        mi_results['Mutual_Information'].quantile(0.90)
            ]
        })
    return mi_stats

mi_statistics = calculate_mi_statistics(mi_results)
print("\nEstatísticas Resumidas do Mutual Information:")
print(mi_statistics)


Estatísticas Resumidas do Mutual Information:
      Estatística       Valor
0  Total Features  122.000000
1           Média    0.073058
2         Mediana    0.008032
3   Desvio Padrão    0.120168
4          Mínimo    0.000000
5          Máximo    0.657781
6        Q1 (25%)    0.000055
7        Q3 (75%)    0.123737
8   Q90 (Top 10%)    0.227615


In [6]:
def create_features_classification_dataset(values, feature_names, metric_name):
    """
    Cria um dataset com classificação das features por faixas de valores
    
    Parâmetros:
    - values: pandas.Series com os valores da métrica
    - feature_names: pandas.Series com os nomes das features
    - metric_name: string com o nome da métrica (ex: 'Mutual_Information')
    
    Retorna:
    - pandas.DataFrame com colunas: Feature, Valor, Faixa
    """
    
    # Definir faixas de análise (mesmas da função analyze_distribution_ranges)
    ranges = [
        (0.0, 0.001, "Muito Baixa"),
        (0.001, 0.01, "Baixa"),
        (0.01, 0.05, "Moderada"),
        (0.05, 0.1, "Alta"),
        (0.1, float('inf'), "Muito Alta")
    ]
    
    # Criar lista para armazenar os dados
    classification_data = []
    
    # Classificar cada feature
    for feature, value in zip(feature_names, values):
        
        # Encontrar a faixa apropriada
        faixa = "Indefinida"  # Valor padrão caso não encontre
        
        for min_val, max_val, label in ranges:
            if max_val == float('inf'):
                if value >= min_val:
                    faixa = label
                    break
            else:
                if min_val <= value < max_val:
                    faixa = label
                    break
        
        # Adicionar à lista de dados
        classification_data.append({
            'Feature': feature,
            'Valor': value,
            'Faixa': faixa
        })
    
    # Criar DataFrame
    df_classification = pd.DataFrame(classification_data)
    
    return df_classification

mi_classification = create_features_classification_dataset(
        values=mi_results['Mutual_Information'],
        feature_names=mi_results['Feature'],
        metric_name='Mutual_Information'
    )

print("\nClassificação das Features por Faixas de Valores:")
mi_classification    





Classificação das Features por Faixas de Valores:


,Feature,Valor,Faixa
0,mean_os_net_bytes_sent,0.657781,Muito Alta
1,mean_os_net_num_connections,0.531574,Muito Alta
2,mean_process_cpu_system,0.404249,Muito Alta
3,mean_process_disk_write_bytes,0.391475,Muito Alta
4,mean_process_cpu_user,0.364710,Muito Alta
...,...,...,...
117,mean_container_net_rx_errs,0.000000,Muito Baixa
118,mean_container_net_tx_colls,0.000000,Muito Baixa
119,mean_container_net_tx_carrier,0.000000,Muito Baixa
120,mean_container_net_tx_drop,0.000000,Muito Baixa


In [7]:
print("=" * 80)
print("CONTAGEM DE FEATURES POR FAIXA")
print("=" * 80)

contagem = mi_classification['Faixa'].value_counts()
print("\n📊 Contagem absoluta:")
print(contagem)

CONTAGEM DE FEATURES POR FAIXA

📊 Contagem absoluta:
Faixa
Muito Baixa    39
Muito Alta     35
Baixa          24
Moderada       16
Alta            8
Name: count, dtype: int64


In [8]:
classified_mi = mi_classification[mi_classification['Faixa'].isin(['Alta', 'Moderada','Baixa'])]
selected_features = classified_mi['Feature'].tolist()
print("\nFeatures Selecionadas (Alta, Moderada e Baixa MI):")
print(len(selected_features))




Features Selecionadas (Alta, Moderada e Baixa MI):
48


In [9]:
lib_analise.print_informacao_analise()

Nome do dataset:  svm
X_train.shape (32259, 122)
X_test.shape (24194, 122)
X_val.shape (24195, 122)
X_train_scaled.shape None
X_test_scaled.shape None
X_val_scaled.shape None
classes_mapping {'interf': np.int64(0), 'normal': np.int64(1)}
features_ganho_informacao ['mean_os_cpu_ctx_switches', 'mean_os_cpu_guest', 'mean_os_cpu_guest_nice', 'mean_os_cpu_idle', 'mean_os_cpu_interrupts', 'mean_os_cpu_iowait', 'mean_os_cpu_irq', 'mean_os_cpu_nice', 'mean_os_cpu_soft_interrupts', 'mean_os_cpu_softirq', 'mean_os_cpu_steal', 'mean_os_cpu_syscalls', 'mean_os_cpu_system', 'mean_os_cpu_user', 'mean_os_disk_discard_io', 'mean_os_disk_discard_merges', 'mean_os_disk_discard_sectors', 'mean_os_disk_discard_ticks', 'mean_os_disk_in_flight', 'mean_os_disk_io_ticks', 'mean_os_disk_read_io', 'mean_os_disk_read_merge', 'mean_os_disk_read_sectors', 'mean_os_disk_read_ticks', 'mean_os_disk_time_in_queue', 'mean_os_disk_write_io', 'mean_os_disk_write_merge', 'mean_os_disk_write_sectors', 'mean_os_disk_write_t

In [10]:
datasets['features_ganho_informacao'] = selected_features
lib_analise.save_informacao_analise(datasets=datasets)
datasets = lib_analise.get_dataset_analise(analise_ganho_de_informacao=True)
lib_analise.save_informacao_analise(datasets=datasets)
print("=>",len(datasets['features_ganho_informacao']))
lib_analise.print_informacao_analise()

✅ Dataset salvo com sucesso em ../dataset/svm.pkl
============> 48
✅ Dataset salvo com sucesso em ../dataset/svm.pkl
=> 48
Nome do dataset:  svm
X_train.shape (32259, 48)
X_test.shape (24194, 48)
X_val.shape (24195, 48)
X_train_scaled.shape None
X_test_scaled.shape None
X_val_scaled.shape None
classes_mapping {'interf': np.int64(0), 'normal': np.int64(1)}
features_ganho_informacao ['mean_container_mem_pgpgin', 'mean_container_net_tx_packets', 'mean_os_mem_nr_mapped', 'mean_container_net_rx_packets', 'mean_os_mem_pgpgout', 'mean_os_disk_write_sectors', 'mean_os_disk_write_io', 'mean_container_cpu_user', 'mean_container_cpu_system', 'mean_container_mem_active_file', 'mean_os_mem_nr_active_file', 'mean_os_disk_time_in_queue', 'mean_container_mem_rss', 'mean_container_mem_inactive_anon', 'mean_container_mem_mapped_file', 'mean_os_disk_write_ticks', 'mean_os_mem_pgmajfault', 'mean_os_mem_pgpgin', 'mean_os_disk_io_ticks', 'mean_os_disk_read_ticks', 'mean_os_mem_nr_inactive_anon', 'mean_os_di